# Heat Resilience in Sight — Code 3: Streetscape heat resilience

Calibrate $V^\mathrm{need}$ on Hong Kong's mean VATA and June–August **hourly open-sun** heat-degree demand, then write monthly/annual SHR tables and the paper figures.

$$\mathrm{SHR}_{c,m}=\bar V_c-\bar V_\mathrm{HK}\cdot D_{c,m}/\bar D_\mathrm{HK,JJA}$$

Demand $D$ is the monthly mean of daytime-maximum $\max(\mathrm{UTCI}-26,0)$. The Hong Kong JJA anchor is normative. Alternate windows, slopes, and shaded demand are in `code_5_robustness_checks.ipynb`.

**Outputs:** `data/outputs/{monthly,annual}_climate_vata.csv`, `data/outputs/calibration.json`, and `figures/fig_*.png` (analysis plots only; `fig_framework` and `fig_mapping` are static assets).

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from scripts.climate import panel_climatology
from scripts.config import FIGS, PANEL_ORDER, TABLES
from scripts.figures import write_paper_figures
from scripts.panel import summarise_panel
from scripts.shr import attach_shr, write_tables

stats = summarise_panel(write=True)
monthly_raw = panel_climatology()
monthly, annual, meta = attach_shr(stats, monthly_raw)
write_tables(monthly, annual, meta)
write_paper_figures(stats, monthly, annual, FIGS)

print('calibration', {k: meta[k] for k in ('calibrate_city', 'v_ref', 'd_ref', 'u0_utci_c')})
annual.loc[PANEL_ORDER, ['short', 'n', 'v_bar', 'utci', 'v_need', 'shr', 'inadequate_frac']].round(3)

## 1. Peak-season check

Northern-hemisphere July and southern-hemisphere January are the months that should not be hidden by an annual mean.

In [ ]:
peak = monthly[
    ((monthly.city.isin(['svi_sg', 'svi_hk', 'svi_tokyo', 'svi_ny'])) & (monthly.month == 7))
    | ((monthly.city.isin(['svi_melbourne', 'svi_rio', 'svi_capetown', 'svi_jh'])) & (monthly.month == 1))
]
peak[['city', 'month', 'utci', 'demand', 'v_need', 'shr', 'inadequate_frac']].round(3)

In [ ]:
print('wrote tables to', TABLES)
print('wrote figures to', FIGS)